# **Tutorial 07 - Sparse Recovery and Basis Pursuit**: Supplementary Interactive Material

-------------------------------------------------------------------------

Sparse recovery seeks a solution with few nonzero coefficients. Here we use a collection of faces as a dictionary and reconstruct a face from a small fraction of its pixels. Along the way, we will see how the $\ell_1$ penalty trades reconstruction accuracy for sparsity.

In [ ]:
import numpy as np
import time
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_olivetti_faces

## From basis pursuit to a penalized problem

For an exact system $Ax=b$, **basis pursuit** minimizes $\|x\|_1$ subject to $Ax=b$. With noisy or incomplete measurements, basis pursuit denoising instead permits a residual:

$$\min_x \|x\|_1 \quad\text{subject to}\quad \|Ax-b\|_2\leq\varepsilon.$$

We will solve the related **penalized** problem

$$\min_x \; \frac{1}{2}\|Ax-b\|_2^2+\lambda\|x\|_1.$$

Larger $\lambda$ penalizes nonzero coefficients more strongly; $\lambda=0$ gives ordinary least squares. The constrained and penalized formulations describe related tradeoffs, but $\varepsilon$ and $\lambda$ are different parameters and are not interchangeable. Sparse coefficients are encouraged, not guaranteed to recover the original signal.

## Example: recovering a face from missing pixels

The Olivetti dataset contains 400 grayscale faces, each with $64\times64$ pixels. Flatten each image into a column of a dictionary $D\in\mathbb{R}^{4096\times399}$, leaving the target face out. We observe only some target pixels. If $\Omega$ is the set of observed pixel indices, the problem above uses $A=D_{\Omega,:}$ and $b=(b_{\rm true})_\Omega$; unobserved pixels do not enter the fit.

The fitted image is $\widehat b=Dx$. The target is excluded from the dictionary, so it must be approximated by other faces. The first call to `fetch_olivetti_faces` may download the dataset.

In [ ]:
faces = fetch_olivetti_faces()
images = faces.data.astype(np.float64)
rng = np.random.default_rng(0)
target_index = rng.integers(len(images))
b_true = images[target_index]
# Keep the original dataset indices for interpreting the coefficients.
dictionary_indices = np.delete(np.arange(len(images)), target_index)
D = images[dictionary_indices].T

missing_fraction = 0.7  # Try 0.3 or 0.9, then rerun this and subsequent cells.
mask = rng.random(b_true.size) >= missing_fraction
A = D[mask, :]
b = b_true[mask]
print(f"Target face: {target_index}; dictionary: {D.shape}; observed pixels: {mask.sum()} / {mask.size}")
print(f"Fitting system A: {A.shape}")

## Compare ISTA and FISTA

Both methods minimize the same objective:

$$f(x)=\underbrace{\frac12\|Ax-b\|_2^2}_{\text{fit the observed pixels}}+\underbrace{\lambda\|x\|_1}_{\text{encourage sparse coefficients}}.$$

The idea is to alternate two simple operations: move toward a better fit, then shrink the coefficients toward zero.

### ISTA: improve the fit, then shrink

The squared-error term $g(x)=\frac12\|Ax-b\|_2^2$ has gradient $\nabla g(x)=A^\top(Ax-b)$. Starting from $x_0=0$, the **iterative shrinkage-thresholding algorithm (ISTA)** repeats

$$\begin{aligned}
z_k&=x_k-\frac1L A^\top(Ax_k-b),&&\text{gradient step},\\
x_{k+1}&=S_{\lambda/L}(z_k),&&\text{soft thresholding},
\end{aligned}$$

where $L=\|A\|_2^2$ is the square of the largest singular value of $A$. The step size $1/L$ keeps the gradient step small enough to guarantee that the full objective does not increase.

Soft thresholding acts on each coefficient separately:

$$S_\tau(z)_i=\operatorname{sign}(z_i)\max(|z_i|-\tau,0).$$

For example, with $\tau=0.2$, the coefficients $(0.1,\,0.8,\,-0.5)$ become $(0,\,0.6,\,-0.3)$. Small coefficients disappear, while larger ones shrink toward zero. A larger $\lambda$ gives a larger threshold and encourages more zeros.

This shrinkage step is the **proximal operator** of the $\ell_1$ penalty: it exactly solves the small optimization problem

$$S_{\lambda/L}(z)=\arg\min_u\left\{\frac L2\|u-z\|_2^2+\lambda\|u\|_1\right\}.$$

Thus ISTA balances staying near the gradient step with reducing the $\ell_1$ penalty.

### FISTA: add momentum

The **fast iterative shrinkage-thresholding algorithm (FISTA)** takes the same gradient-and-threshold step at an extrapolated point $y_k$. Starting with $x_0=y_0=0$ and $t_0=1$, it uses

$$\begin{aligned}
x_{k+1}&=S_{\lambda/L}\!\left(y_k-\frac1L A^\top(Ay_k-b)\right),\\
t_{k+1}&=\frac{1+\sqrt{1+4t_k^2}}{2},\\
y_{k+1}&=x_{k+1}+\frac{t_k-1}{t_{k+1}}(x_{k+1}-x_k).
\end{aligned}$$

The last line adds **momentum** in the direction of the previous update. This often helps FISTA move faster through regions where ISTA makes slow progress. Unlike ISTA, FISTA can temporarily increase the objective.

For this convex problem, the standard bounds on the objective gap $f(x_k)-f(x^\star)$ decrease as $O(1/k)$ for ISTA and $O(1/k^2)$ for FISTA, where $x^\star$ is a minimizer. These describe convergence of the optimization objective, not reconstruction accuracy.

### What the comparison measures

Both implementations check whether one more ISTA step would change the current solution appreciably. Writing

$$T(x)=S_{\lambda/L}\!\left(x-\frac1L A^\top(Ax-b)\right),$$

the reported residual is

$$r(x)=\frac{\|x-T(x)\|_2}{\max(1,\|x\|_2)}.$$

At an optimum, $x=T(x)$, so this residual is zero. Both methods stop when it falls below `tol`, or when they reach `max_iter`.

We compare the same $A,b,\lambda$, starting point, tolerance, and iteration limit. Look at iteration counts and convergence flags as well as elapsed time: these live timings include setup and can vary with machine load.

In [ ]:
def soft_threshold(z, threshold):
    return np.sign(z) * np.maximum(np.abs(z) - threshold, 0.0)


def ista(A, b, lam, max_iter=3000, tol=1e-6):
    """Minimize 0.5 * ||A x - b||^2 + lam * ||x||_1 with ISTA."""
    if lam < 0:
        raise ValueError("lam must be nonnegative")
    gram = A.T @ A
    rhs = A.T @ b
    L = np.linalg.eigvalsh(gram)[-1]
    if L <= 0:
        raise ValueError("The observed dictionary must contain nonzero entries.")
    x = np.zeros(A.shape[1])
    for iteration in range(1, max_iter + 1):
        x_new = soft_threshold(x - (gram @ x - rhs) / L, lam / L)
        prox = soft_threshold(x_new - (gram @ x_new - rhs) / L, lam / L)
        residual = np.linalg.norm(x_new - prox) / max(1.0, np.linalg.norm(x_new))
        if residual < tol:
            return x_new, iteration, residual, True
        x = x_new
    return x_new, iteration, residual, False


def fista(A, b, lam, max_iter=3000, tol=1e-6):
    """Minimize 0.5 * ||A x - b||^2 + lam * ||x||_1 with FISTA."""
    if lam < 0:
        raise ValueError("lam must be nonnegative")
    gram = A.T @ A
    rhs = A.T @ b
    L = np.linalg.eigvalsh(gram)[-1]
    if L <= 0:
        raise ValueError("The observed dictionary must contain nonzero entries.")
    x = np.zeros(A.shape[1])
    y = x.copy()
    t = 1.0
    for iteration in range(1, max_iter + 1):
        x_new = soft_threshold(y - (gram @ y - rhs) / L, lam / L)
        prox = soft_threshold(x_new - (gram @ x_new - rhs) / L, lam / L)
        residual = np.linalg.norm(x_new - prox) / max(1.0, np.linalg.norm(x_new))
        if residual < tol:
            return x_new, iteration, residual, True
        t_new = (1.0 + np.sqrt(1.0 + 4.0 * t * t)) / 2.0
        y = x_new + ((t - 1.0) / t_new) * (x_new - x)
        x, t = x_new, t_new
    return x_new, iteration, residual, False


lam = 2.0  # Try 0.1, 2, 10, or 100 and rerun this and subsequent cells.
tol = 1e-6
max_iter = 3000
results = {}
timings = {}
for name, solver in [("ISTA", ista), ("FISTA", fista)]:
    start = time.perf_counter()
    results[name] = solver(A, b, lam, max_iter=max_iter, tol=tol)
    timings[name] = time.perf_counter() - start

for name in ["ISTA", "FISTA"]:
    x, n_iter, pg_residual, converged = results[name]
    print(f"{name:5s}: {timings[name]:.4f} s; {n_iter} iterations; "
          f"proximal residual {pg_residual:.2e}; converged={converged}")

# Use the FISTA solution for the reconstruction and subsequent plots.
x_hat, iterations, residual, converged = results["FISTA"]
b_recon = D @ x_hat
print(f"FISTA coefficients with |x| > 1e-4: {np.count_nonzero(np.abs(x_hat) > 1e-4)} / {len(x_hat)}")

## Inspect the reconstruction

The reconstruction below is $Dx$ at **every** pixel; observed values have not been pasted back in. Since the original image is known in this demonstration, we can also measure error on the held-out pixels. In a real recovery problem those pixels would be unknown. Gray areas in the middle panel indicate missing pixels.

In [ ]:
observed_image = np.where(mask, b_true, np.nan).reshape(64, 64)
fig, axes = plt.subplots(1, 3, figsize=(10, 3.3))
cmap = plt.get_cmap('gray').copy()
cmap.set_bad('lightgray')
for ax, image, title in zip(axes, [b_true.reshape(64, 64), observed_image,
                                   b_recon.reshape(64, 64)],
                            ['Original', 'Observed pixels', 'Reconstruction']):
    ax.imshow(image, cmap=cmap, vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.show()

print(f"Observed-pixel RMSE: {np.sqrt(np.mean((b_recon[mask] - b_true[mask])**2)):.4f}")
if (~mask).any():
    print(f"Missing-pixel RMSE:  {np.sqrt(np.mean((b_recon[~mask] - b_true[~mask])**2)):.4f}")
else:
    print("Missing-pixel RMSE: not defined (no pixels are missing).")

## Which faces contribute?

Each coefficient corresponds to a face in the original dataset, except for the omitted target. Coefficients can be negative: this is an unconstrained linear combination, not a weighted average. The plot highlights the 15 largest coefficients by magnitude.

In [ ]:
top = np.argsort(np.abs(x_hat))[-15:][::-1]
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(np.arange(len(top)), x_hat[top])
ax.set_xticks(np.arange(len(top)))
ax.set_xticklabels(dictionary_indices[top], rotation=45)
ax.set_xlabel('Original dataset face index')
ax.set_ylabel('Coefficient')
ax.set_title('15 largest coefficients by magnitude')
ax.axhline(0, color='black', linewidth=0.7)
plt.tight_layout()
plt.show()

> **Explore:** Vary $\lambda$ and rerun the solver and plots. How do the number of nonzero coefficients, observed-pixel error, and missing-pixel error change? Does the smallest observed error also give the best missing-pixel reconstruction?
>
> Change the missing fraction and rerun from the data-preparation cell. How much information can be removed before the reconstruction loses recognizable features? The mask and target are fixed by the random seed for reproducible comparisons.